In [ ]:
from functools import partial
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch
import yaml
import matplotlib.pyplot as plt

sys.path.append("../")
from src.data_generation.grid import Grid
from src.real_data.dataloader import temporal_split, build_task, make_real_eval_set
from src.model.finetune import finetune
from src.model.quote_loss import quote_arb_loss
from src.model.preprocessed_dataset import preprocess_surfaces
from src.evaluation import surface_eval as SE
from src.evaluation.surface_eval import check_arbitrage_flat

cfg = yaml.safe_load(open("../config.yaml"))
g = Grid(cfg)
n_grid = len(g.z)

In [ ]:
START = "2023-01-01"
train_s, val_s, test_s = temporal_split(START, val_months=1, test_months=3)

day = lambda s: s["date"].iloc[0].date()
for name, p in [("train", train_s), ("val", val_s), ("test", test_s)]:
    print(f"{name:5s} {len(p):3d} surfaces   {day(p[0])} -> {day(p[-1])}")

In [ ]:
RUN_NAME = "real_v1"
N_CONTEXT = (10, 40)
N_HELDOUT = 1500 
EVAL_SIZES = [5, 10, 20, 40]
GROUP_SIZE = 1

N_EPOCHS = 3
N_PER_EPOCH = 8
BATCH_SIZE = 8
VAL_EVERY = 25
LR = 1e-5

LAMBDA_CAL = 1.0
LAMBDA_BF = 1.0

In [ ]:
train_provider = partial(build_task, train_s, n_context=N_CONTEXT, grid=g, n_heldout=N_HELDOUT, size_group=GROUP_SIZE)
val_data = make_real_eval_set(val_s, EVAL_SIZES, grid=g,n_heldout=N_HELDOUT)
loss_fn = partial(quote_arb_loss, grid_shape=g.shape, lambda_cal=LAMBDA_CAL, lambda_bf=LAMBDA_BF)

In [ ]:
model = finetune(
    train_provider,
    RUN_NAME,
    n_epochs=N_EPOCHS,
    n_surfaces_per_epoch=N_PER_EPOCH,
    batch_size=BATCH_SIZE,
    val_data=val_data,
    val_every=VAL_EVERY,
    loss_fn=loss_fn,
    lr=LR,
)

In [ ]:
QS = [0.1, 0.25, 0.75, 0.9]


def run_predictions(state, train_list, test_list):
    est = SE._get_eval_estimator()
    est.model_.load_state_dict(state if state is not None else SE._pretrained_state)
    surfaces = preprocess_surfaces(est, train_list, test_list, np.random.default_rng(0), group_size=1)
    recs = []
    with torch.no_grad():
        for i, s in enumerate(surfaces):
            print(f"task {i+1}/{len(surfaces)}", end="\r", flush=True)
            est.raw_space_bardist_ = s.raw_space_bardist
            est.znorm_space_bardist_ = s.znorm_space_bardist
            est.fit_from_preprocessed(s.X_context, s.y_context, s.cat_indices, s.configs,
                                      performance_options=SE._PERF, no_refit=True)
            _, ple, _ = est.forward(s.X_query, use_inference_mode=False)
            L = torch.stack(ple, dim=2)
            Q, B, E, Ld = L.shape
            LBQ = L.permute(1, 2, 0, 3).reshape(B * E, Q, Ld).cpu()
            for gi in range(B):
                bd, lg = s.raw_bardists[gi], LBQ[gi * E:(gi + 1) * E]
                recs.append(dict(
                    mean=bd.mean(lg)[0].numpy(),
                    std=bd.variance(lg)[0].sqrt().numpy(),
                    q={p: bd.icdf(lg, p)[0].numpy() for p in QS},
                    y_te=s.y_query_raw[gi].numpy(),
                    zt=s.X_query_raw[gi][:, :2].numpy(),
                ))
    return recs


state = torch.load(Path("../checkpoints") / RUN_NAME / "final.pt", map_location="cpu")
et_tr, et_te = make_real_eval_set(test_s, EVAL_SIZES, grid=g)
recs = run_predictions(state, et_tr, et_te)
for r, (_, y_ctx) in zip(recs, et_tr):
    r["n"] = len(y_ctx) // 2
print(len(recs), "test tasks")

In [ ]:
# MAE to market mid (%), by context size
rows = []
for r in recs:
    m = np.isfinite(r["y_te"]).all(1)
    mid = r["y_te"][m].mean(1)
    rows.append((r["n"], np.abs(r["mean"][m] - mid).mean() * 100))
print(pd.DataFrame(rows, columns=["n", "mae"]).groupby("n")["mae"].mean().round(4).to_string())

In [ ]:
rows = []
for r in recs:
    m = np.isfinite(r["y_te"]).all(1)
    inside = (r["mean"][m] >= r["y_te"][m, 0]) & (r["mean"][m] <= r["y_te"][m, 1])
    rows.append((r["n"], inside.mean()))
print(pd.DataFrame(rows, columns=["n", "inside"]).groupby("n")["inside"].mean().round(3).to_string())

In [ ]:
# arbitrage violation rate
rows = []
for r in recs:
    cal, bf = check_arbitrage_flat(cfg, r["mean"][:n_grid])
    rows.append((r["n"], cal, bf))
print(pd.DataFrame(rows, columns=["n", "cal", "bf"]).groupby("n")[["cal", "bf"]].mean().round(4).to_string())

In [ ]:
# uncertainty: mean predictive std (%) and central-interval coverage of the mid, by context size
LEVELS = {0.5: (0.25, 0.75), 0.8: (0.1, 0.9), 0.9: (0.05, 0.95), 0.95: (0.025, 0.975)}
rows = []
for r in recs:
    m = np.isfinite(r["y_te"]).all(1)
    mid = r["y_te"][m].mean(1)
    row = {"n": r["n"], "std%": r["std"][m].mean() * 100}
    for lv, (lo, hi) in LEVELS.items():
        row[f"cov{int(lv * 100)}"] = ((mid >= r["q"][lo][m]) & (mid <= r["q"][hi][m])).mean()
    rows.append(row)
print(pd.DataFrame(rows).groupby("n").mean().round(3).to_string())

In [ ]:
r = recs[0]
iv = r["mean"][:n_grid].reshape(g.shape)
m = np.isfinite(r["y_te"]).all(1)
zq, tauq = r["zt"][m, 0], r["zt"][m, 1]
midq = r["y_te"][m].mean(1)

fig, ax = plt.subplots(figsize=(8, 5))
pc = ax.pcolormesh(g.zs, g.ttms, iv, shading="auto", cmap="viridis", alpha=0.9)
ax.scatter(zq, tauq, c=midq, cmap="viridis", s=6, edgecolor="k", linewidth=0.15)
fig.colorbar(pc, label="IV")
ax.set(xlabel="z", ylabel="ttm", title=f"predicted surface + held quotes (n={r['n']})")
plt.show()